# Tool-paper experiments (R1-R5): ranking, calibration, prediction, reliability, practicality

Native pipeline with progress bars. R1 (blinded causal ranking) and R2
(known-circuit benchmark) run in-process; R3-R5 drivers stream their own
tqdm output. Resume-friendly: finished CSVs are skipped.

Models: Llama-3.2-1B/3B/3B-Instruct, Llama-3.1-8B, Qwen2.5-3B/7B
(OLMo omitted by default; uncomment to include). Requires the parametric
cohorts + obstruction/bottleneck CSVs from `run_paper.ipynb`.

In [ ]:
MODELS = [
    "meta-llama/Llama-3.2-1B",
    "meta-llama/Llama-3.2-3B",
    "meta-llama/Llama-3.2-3B-Instruct",
    "meta-llama/Llama-3.1-8B",
    "Qwen/Qwen2.5-3B",
    "Qwen/Qwen2.5-7B",
    # "allenai/OLMo-7B-hf",
]
ONLY        = ""      # substring filter; "" = all
DEVICE      = "cuda"
N_EXAMPLES  = 50      # per model for R1 (half failures, half correct)
FULL_SMALL  = True    # measure ALL LxH candidates on 1B/3B models

import os, gc, json, sys, subprocess, random, torch, pandas as pd
from tqdm.auto import tqdm
torch.set_grad_enabled(False)
pd.set_option("display.precision", 3)

from frozen_cache import Weights, build_cache, certify_frozen
from repairs import certify_wrappers, margin
from ranking import (score_candidates, ground_truth, candidate_subset,
                     ranking_metrics, magnitude_metrics,
                     paired_bootstrap, cumulative_curves, RANKERS)
from frozen_cache import tdla_edge_scores

models = [m for m in MODELS if ONLY.lower() in m.lower()]
tag = lambda m: m.split("/")[-1].replace(".", "").lower()

_loaded = {}
def get_model(name):
    if name in _loaded: return _loaded[name]
    import __main__
    for v in ("model", "tok", "W", "C"):
        if hasattr(__main__, v): delattr(__main__, v)
    _loaded.clear(); gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tk = AutoTokenizer.from_pretrained(name)
    md_ = AutoModelForCausalLM.from_pretrained(
        name, torch_dtype=torch.float32, low_cpu_mem_usage=True,
        attn_implementation="eager").to(DEVICE).eval()
    _loaded[name] = (md_, tk, Weights(md_))
    return _loaded[name]
print(f"{len(models)} models:", [tag(m) for m in models])

def ensure_cohort(m):
    """Build the parametric cohort in place if missing (fresh machine).
    R1 needs only the correct/incorrect split, so no verdict labeling here;
    prefer copying cohorts from the previous run for comparability."""
    path = f"cohorts/{tag(m)}_parametric.jsonl"
    if os.path.exists(path):
        return path
    os.makedirs("cohorts", exist_ok=True)
    from build_cohort import process, load_popqa
    model, tok, W = get_model(m)
    src = load_popqa(170)
    out = []
    bar = tqdm(src, desc=f"{tag(m)} build cohort")
    nf = 0
    for r in bar:
        try:
            rec = process(model, tok, DEVICE, r["prompt"], r["subject"],
                          r["answer"], r.get("aliases", []))
        except (ValueError, IndexError):
            continue
        out.append(rec)
        nf += not rec["correct"]
        bar.set_postfix(failures=nf)
        if nf >= 60 and len(out) - nf >= 60:
            break
    clean = [o for o in out if not o["copy_contaminated"]]
    fails = [o for o in clean if not o["correct"]][:60]
    corrs = [o for o in clean if o["correct"]][:60]
    with open(path, "w") as f:
        for o in fails + corrs:
            f.write(json.dumps(o) + "\n")
    print(f"  built {path}: {len(fails)} failures / {len(corrs)} successes")
    return path

## R1 — Blinded causal ranking (+ specificity)
Rankings frozen from one cached pass; ground truth = live single-candidate
ablations. Small models: all LxH candidates; big models: top-20-per-ranker
union + 40 random.

In [ ]:
import csv
K_GRID = (1, 2, 4, 8, 16, 32)
for m in models:
    out = f"results/{tag(m)}"; os.makedirs(out, exist_ok=True)
    if os.path.exists(f"{out}/ranking.csv"): continue
    model, tok, W = get_model(m)
    full = FULL_SMALL and W.L * W.H <= 800
    cohort = ensure_cohort(m)
    records = [json.loads(l) for l in open(cohort)
               if json.loads(l)["competitor_token"] != -1]
    rng = random.Random(0)
    fails = [r for r in records if r["verdict"] != "correct"]
    corrs = [r for r in records if r["verdict"] == "correct"]
    sample = (rng.sample(fails, min(N_EXAMPLES//2, len(fails))) +
              rng.sample(corrs, min(N_EXAMPLES - N_EXAMPLES//2, len(corrs))))
    rows, spec = [], []
    for i, r in enumerate(tqdm(sample, desc=f"{tag(m)} R1")):
        ids = tok(r["prompt"], return_tensors="pt").input_ids[0].to(DEVICE)
        C = build_cache(model, W, ids); certify_frozen(W, C)
        if i == 0: certify_wrappers(model, W, ids)
        g, c = r["target_first_token"], r["competitor_token"]
        S = list(range(r["source_span"][0], r["source_span"][1] + 1))
        m0, _ = margin(model, ids, g, c)
        scores = score_candidates(model, W, C, ids, g, c, S, seed=i)
        cands = ([(l, h) for l in range(W.L) for h in range(W.H)] if full
                 else candidate_subset(scores, seed=i))
        bar = tqdm(total=len(cands), leave=False, desc="ablate")
        eff = ground_truth(model, W, ids, g, c, S, cands, m0, progress=bar)
        bar.close()
        row = dict(idx=i, verdict=r["verdict"], n_cands=len(cands), m0=m0)
        row.update(ranking_metrics(scores, eff))
        row.update(magnitude_metrics(scores, eff))
        row.update(cumulative_curves(model, W, ids, g, c, S, scores, m0,
                                     effects=eff, ks=K_GRID))
        rows.append(row)
        # specificity: rescore affine under corrupted configs, same ground truth
        wt = rng.choice([x["target_first_token"] for x in records
                         if x["target_first_token"] != g])
        for nm, sc in (("wrong_target",
                        tdla_edge_scores(W, C, wt, S, tok_c=c).sum(-1).cpu()),
                       ("random_target",
                        tdla_edge_scores(W, C, rng.randrange(W.WU.shape[0]),
                                         S, tok_c=c).sum(-1).cpu())):
            spec.append(dict(idx=i, config=nm,
                             rho=ranking_metrics({"x": sc}, eff)["x_rho"]))
        spec.append(dict(idx=i, config="true", rho=row["affine_rho"]))
    pd.DataFrame(rows).to_csv(f"{out}/ranking.csv", index=False)
    pd.DataFrame(spec).to_csv(f"{out}/specificity.csv", index=False)
    df = pd.DataFrame(rows)
    print(f"== {tag(m)} == median rho: " + "  ".join(
        f"{n}={df[f'{n}_rho'].median():+.2f}" for n in RANKERS))

In [ ]:
# ---- R1 summary: Spearman table + cumulative curves ----
import glob, matplotlib.pyplot as plt, numpy as np
frames = [pd.read_csv(p).assign(model=p.split(os.sep)[-2])
          for p in glob.glob("results/*/ranking.csv")]
if frames:
    R = pd.concat(frames)
    tab = pd.DataFrame({n: R.groupby("model")[f"{n}_rho"].median()
                        for n in RANKERS})
    display(tab.round(3))
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
    for n in list(RANKERS) + ["oracle"]:
        ks = [int(c.split("@")[1]) for c in R.columns
              if c.startswith(f"{n}_drop@")]
        if not ks: continue
        ks = sorted(ks)
        ax[0].plot(ks, [R[f"{n}_drop@{k}"].mean() for k in ks],
                   marker="o", label=n)
        if n != "oracle" and f"oracle_drop@{ks[0]}" in R:
            ax[1].plot(ks, [(R[f"oracle_drop@{k}"] -
                             R[f"{n}_drop@{k}"]).mean() for k in ks],
                       marker="o", label=n)
    ax[0].set_xlabel("k ablated"); ax[0].set_ylabel("mean margin drop")
    ax[0].set_title("cumulative drop"); ax[0].legend(fontsize=8)
    ax[1].set_xlabel("k"); ax[1].set_ylabel("regret vs oracle")
    ax[1].set_title("top-k regret"); ax[1].legend(fontsize=8)
    plt.tight_layout(); plt.savefig("results/R1_ranking.png", dpi=150)
    plt.show()
    spec = pd.concat([pd.read_csv(p) for p in
                      glob.glob("results/*/specificity.csv")])
    print("specificity (affine rho by config):")
    display(spec.groupby("config").rho.median().round(3))

## R2 — Known-circuit benchmark (planted decoy route)
Tiny model, trains in minutes; three seeds.

In [ ]:
from known_circuit import run as kc_run
kc = []
for seed in (0, 1, 2):
    out = f"results/known_circuit_seed{seed}"
    if os.path.exists(f"{out}/known_circuit.csv"):
        kc.append(pd.read_csv(f"{out}/known_circuit.csv").assign(seed=seed))
        continue
    summary, rows = kc_run(out, steps=2500, n_eval=150, device=DEVICE,
                           seed=seed)
    kc.append(pd.DataFrame(rows).assign(seed=seed))
K = pd.concat(kc)
print(f"pooled n={len(K)}: ID@1 affine {K.affine_hit.mean():.2f} vs "
      f"attention {K.attn_hit.mean():.2f}; attention-on-sink "
      f"{K.attn_top_is_sink.mean():.2f}")
print(f"median drop: true col {K.drop_true.median():+.2f}, affine-top "
      f"{K.drop_affine_top.median():+.2f}, attn-top "
      f"{K.drop_attn_top.median():+.2f}")

## R3-R5 — predictive value, reliability, practicality
Drivers stream their own progress; per-model, resume-friendly.

In [ ]:
PY = sys.executable
for m in models:
    t = tag(m); out = f"results/{t}"
    cohort = f"cohorts/{t}_parametric.jsonl"
    for script, artifact in (("run_predictive.py", "predictive.csv"),
                             ("run_reliability.py", "reliability.csv"),
                             ("run_profile.py", "profile.csv")):
        if os.path.exists(f"{out}/{artifact}"):
            continue
        if not os.path.exists(cohort):
            cohort = ensure_cohort(m)
        if script == "run_predictive.py" and not os.path.exists(
                f"{out}/obstruction.csv"):
            print(f"  skip predictive for {t}: needs "
                  f"{out}/obstruction.csv from run_paper.ipynb")
            continue
        print(f"--- {script} {t} ---")
        subprocess.run([PY, script, "--model", m, "--cohort", cohort,
                        "--out", out, "--device", DEVICE])

In [ ]:
# ---- pooled summaries for R3-R5 ----
for name in ("predictive", "reliability", "profile"):
    ps = glob.glob(f"results/*/{name}.csv")
    if not ps: continue
    df = pd.concat([pd.read_csv(p).assign(model=p.split(os.sep)[-2])
                    for p in ps])
    print(f"==== {name} ====")
    if name == "predictive":
        display(df.pivot_table(index="tier", columns="target",
                               values="score").round(3))
    elif name == "reliability":
        display(df.groupby("kind")[["map_cos", "topk_jac"]].median().round(3))
    else:
        display(df.groupby("model")[["c2_err", "x_cache", "x_ob", "x_tdla",
                                     "peak_gb"]].median().round(3))

## R1' — magnitude-aware summary + paired statistics
Oracle-fraction, nDCG, oracle recall; paired bootstrap CIs for affine minus
each baseline (rho, capture@8), win rates. Rerun the R1 cell first if
ranking.csv predates the gradxact sign audit (`rm results/*/ranking.csv`).

In [ ]:
import glob
frames = [pd.read_csv(p).assign(model=p.split(os.sep)[-2])
          for p in glob.glob("results/*/ranking.csv")]
if frames:
    R = pd.concat(frames)
    for metric in ("capture@8", "ndcg@8", "orecall@8"):
        cols = {n: f"{n}_{metric}" for n in RANKERS
                if f"{n}_{metric}" in R.columns}
        if not cols:
            print(f"{metric}: not in CSVs — rerun R1 after "
                  f"`rm results/*/ranking.csv` (old files also carry the "
                  f"pre-audit gradxact sign)")
            continue
        print(f"-- {metric} (median per model, equal-weight avg) --")
        t = pd.DataFrame({n: R.groupby("model")[c].median()
                          for n, c in cols.items()})
        t.loc["AVG"] = t.mean()
        display(t.round(3))
    print("-- paired affine minus baseline (pooled examples) --")
    for base in ("gradxact", "dla", "attention", "actnorm"):
        for met in ("rho", "capture@8"):
            ca, cb = f"affine_{met}", f"{base}_{met}"
            if ca not in R.columns or cb not in R.columns:
                continue
            sub = R[[ca, cb]].dropna()
            if len(sub) < 5:
                continue
            pb = paired_bootstrap(sub[ca].tolist(), sub[cb].tolist())
            print(f"  vs {base:<9s} {met:<10s} diff "
                  f"{pb['mean_diff']:+.3f} [{pb['ci_lo']:+.3f},"
                  f"{pb['ci_hi']:+.3f}]  win {pb['win_rate']:.2f}")

## R2' — known-circuit v2 (factorial, 12 seeds)
Easy / decoy / competing / hard. Decoy = synthetic high-attention edge with
zero target OV, exact by construction (the sink-resistance test v1 lacked).

In [ ]:
if not os.path.exists("results/known_circuit2/known_circuit2.csv"):
    subprocess.run([PY, "known_circuit2.py", "--out",
                    "results/known_circuit2", "--seeds", "12",
                    "--device", DEVICE])
else:
    print("known_circuit2.csv exists; delete to rerun")

## R4' — reliability v2 (verified routes, discrimination AUC)
and the R(S) stress sweep.

In [ ]:
for m in models:
    t = tag(m); out = f"results/{t}"
    cohort = f"cohorts/{t}_parametric.jsonl"
    if not os.path.exists(cohort):
        cohort = ensure_cohort(m)
    if not os.path.exists(f"{out}/reliability2.csv"):
        subprocess.run([PY, "run_reliability2.py", "--model", m,
                        "--cohort", cohort, "--out", out,
                        "--device", DEVICE])
    if not os.path.exists(f"{out}/ob_stress.csv"):
        subprocess.run([PY, "run_ob_stress.py", "--model", m,
                        "--out", out, "--device", DEVICE])

In [ ]:
# pooled reliability-v2 discrimination + stress summary
for name, key in (("reliability2", "rbo"), ("ob_stress", "ob")):
    ps = glob.glob(f"results/*/{name}.csv")
    if not ps: continue
    df = pd.concat([pd.read_csv(p).assign(model=p.split(os.sep)[-2])
                    for p in ps])
    print(f"==== {name} ====")
    if name == "reliability2":
        from run_obstruction_validation import auc
        for mt in ("affine", "attention", "dla", "actnorm"):
            sub = df[df["map"] == mt]
            chg = (1 - sub[sub.kind == "route"].rbo).tolist()
            nch = (1 - sub[sub.kind == "nuisance"].rbo).tolist()
            if chg and nch:
                print(f"  {mt:<10s} pooled AUC(1-RBO) {auc(chg, nch):.3f}")
        print("  route verification rate:",
              round((df.kind == "route").sum() /
                    max((df.kind.isin(["route", "route_unverified"])).sum(),
                        1), 2))
    else:
        display(df.groupby(["n_pairs", "ambiguous"])
                .agg(acc=("correct", "mean"),
                     ob_nz=("ob", lambda s: (s > 1e-9).mean()),
                     ob_med=("ob", "median")).round(4))

## Full results dump — print everything, save to Excel/HTML/zip

In [ ]:
# ---------- tool-paper results: print everything, save nicely ----------
import glob, os, pandas as pd
from datetime import date
from ranking import RANKERS

OUT = "results/full_report_tool"
os.makedirs(OUT, exist_ok=True)
STAGES = ("ranking", "specificity", "known_circuit", "predictive",
          "reliability", "profile")
tables = []
for p in sorted(glob.glob("results/*/*.csv")):
    model, stage = p.split(os.sep)[1], p.split(os.sep)[2].replace(".csv", "")
    if stage in STAGES:
        tables.append((model, stage, pd.read_csv(p)))
print(f"{len(tables)} tool-paper tables found")
for model, stage, df in tables:
    print(f"\n{'='*70}\n### {model} — {stage}   ({len(df)} rows)\n{'='*70}")
    with pd.option_context("display.max_rows", None,
                           "display.max_columns", None,
                           "display.width", 250, "display.precision", 3):
        display(df)

def pool(stage):
    fs = [df.assign(model=m) for m, s, df in tables if s == stage]
    return pd.concat(fs) if fs else None

print("\n" + "#" * 70 + "\n### HEADLINE SUMMARIES\n" + "#" * 70)
R = pool("ranking")
if R is not None:
    print("\n[R1] median Spearman per ranker:")
    display(pd.DataFrame({n: R.groupby("model")[f"{n}_rho"].median()
                          for n in RANKERS if f"{n}_rho" in R}).round(3))
    for split, sub in R.groupby(R.verdict.eq("correct")
                                 .map({True: "correct", False: "failures"})):
        print(f"  {split}: affine rho median "
              f"{sub.affine_rho.median():+.3f} (n={len(sub)})")
S = pool("specificity")
if S is not None:
    print("\n[R1b] specificity — affine rho by config:")
    display(S.groupby("config").rho.median().round(3))
K = pool("known_circuit")
if K is not None:
    print(f"\n[R2] known-circuit (n={len(K)}): ID@1 affine "
          f"{K.affine_hit.mean():.2f} vs attn {K.attn_hit.mean():.2f}; "
          f"attn-top-is-sink {K.attn_top_is_sink.mean():.2f}")
    print(f"     drops: true {K.drop_true.median():+.2f} affine-top "
          f"{K.drop_affine_top.median():+.2f} attn-top "
          f"{K.drop_attn_top.median():+.2f}")
P = pool("predictive")
if P is not None:
    print("\n[R3] held-out predictive value:")
    display(P.pivot_table(index="tier", columns="target",
                          values="score").round(3))
L = pool("reliability")
if L is not None:
    print("\n[R4] reliability:")
    display(L.groupby("kind")[["map_cos", "topk_jac", "d_ob"]]
             .median().round(3))
F = pool("profile")
if F is not None:
    print("\n[R5] practicality:")
    display(F.groupby("model").agg(
        c2_pass=("c2_pass", "mean"), c2_err_med=("c2_err", "median"),
        c2_err_max=("c2_err", "max"), x_cache=("x_cache", "median"),
        x_ob=("x_ob", "median"), x_tdla=("x_tdla", "median"),
        peak_gb=("peak_gb", "max")).round(3))

try:
    with pd.ExcelWriter(f"{OUT}/tool_results.xlsx", engine="openpyxl") as xw:
        for m, s, df in tables:
            df.to_excel(xw, sheet_name=f"{m[:18]}_{s[:11]}"[:31], index=False)
    print(f"\nExcel -> {OUT}/tool_results.xlsx")
except ImportError:
    print("pip install openpyxl for the Excel export")
css = ("<style>body{font-family:sans-serif;margin:2em}"
       "table{border-collapse:collapse;font-size:12px;margin:1em 0}"
       "td,th{border:1px solid #ccc;padding:3px 8px;text-align:right}"
       "th{background:#f0f0f0}h2{border-bottom:2px solid #444}"
       "tr:nth-child(even){background:#fafafa}</style>")
parts = [f"<html><head>{css}</head><body><h1>Tool-paper results "
         f"({date.today().isoformat()})</h1>"]
for m, s, df in tables:
    parts.append(f"<h2>{m} — {s} ({len(df)} rows)</h2>")
    parts.append(df.to_html(index=False, float_format=lambda v: f"{v:.3f}"))
parts.append("</body></html>")
open(f"{OUT}/tool_results.html", "w").write("\n".join(parts))
print(f"HTML  -> {OUT}/tool_results.html")
import shutil
print("zip   ->", shutil.make_archive("tool_results_bundle", "zip", "results"))